___
# <center>Atividade: Tipos de Variáveis</center>
___

## Aula 02

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * identificar o tipo de cada variável de uma base jurídica;
 * distinguir o *dtype* do pandas da natureza estatística da variável;
 * converter uma coluna de um tipo para outro;
 * declarar variáveis categóricas, com e sem ordem.

**Como este notebook funciona:** cada operação nova aparece primeiro resolvida e
comentada. Logo depois vem um bloco **✍️ Agora você**, com uma célula em que
faltam pedaços, marcados por `________`, para você completar com a mesma
operação em outra coluna.


___
<div id="indice"></div>

## Índice

- [Processos de saúde no TJSP](#problema)

- [Tipos de variáveis](#tipos)
    - [EXERCÍCIO 1: classifique as variáveis](#ex1)

- [O que o pandas achou de cada coluna](#primeiro-olhar)
    - [🔎 Olhando o dtype e a contagem de valores distintos](#info)
    - [⚠️ O dtype não é o tipo da variável](#dtype)
    - [🚩 Variável que não varia](#constante)

- [Convertendo o tipo de uma coluna](#converter)
    - [🔤 De número para texto](#astype)
    - [📅 De texto para data](#datas)
    - [➖ Subtraindo duas datas](#subtracao)
    - [EXERCÍCIO 2: o tempo até a última movimentação](#ex2)

- [Variáveis categóricas](#categoricas)
    - [🏷️ Declarando a variável como categórica](#categorical)
    - [🔢 Declarando a ordem das categorias](#ordinal)
    - [🕳️ Valor faltante e a categoria Outros](#outros)

- [✂️ De numérica para categórica com pd.cut](#cut)
    - [EXERCÍCIO 3: quantos passaram de seis meses](#ex3)

- [Respondendo à pergunta](#resposta)
    - [EXERCÍCIO 4: uma variável derivada](#ex4)

- [RESUMO](#resumo)


___
<div id="problema"></div>

# Processos de saúde no TJSP

Uma pesquisa empírica em Direito termina numa tabela. Quem decide o que vira
coluna dessa tabela é você, e cada coluna é uma **variável**.

A pergunta que vamos perseguir hoje é esta:

> Entre 2023 e 2025, como se distribuem no TJSP os processos com assunto de
> saúde (medicamento, tratamento médico-hospitalar, plano de saúde): em que grau
> tramitam, de que classe são, e quanto tempo passa entre o ajuizamento e a
> última movimentação registrada?

Repare que a pergunta já obriga a decidir coisas. "Quanto tempo" pede uma
variável numérica que **não existe** na base e vai precisar ser construída.

**As variáveis da base têm os seguintes significados:**

* `numero_processo`: identificador do processo no CNJ.
* `tribunal`: sigla do tribunal.
* `grau`: instância em que o processo tramita (G1, G2 ou JE).
* `classe` e `classe_codigo`: tipo de ação, por extenso e em código.
* `assunto` e `n_assuntos`: matéria discutida e quantos assuntos o processo tem.
* `orgao_julgador`: vara ou câmara responsável.
* `municipio_ibge`: código IBGE do município.
* `sistema`: sistema processual em que o processo corre (SAJ, Projudi, PJe).
* `formato`: eletrônico ou físico.
* `nivel_sigilo`: grau de segredo de justiça.
* `data_ajuizamento`: data em que o processo foi distribuído.
* `data_ultima_atualizacao`: data da última movimentação registrada.

A base saiu da API pública do DataJud (CNJ), pela biblioteca
[juscraper](https://github.com/jtrecenti/juscraper). O código abaixo foi rodado
uma vez e está aqui só para você conhecer a origem da tabela. **Não precisa
rodar.**

```python
import juscraper as jus

datajud = jus.scraper("datajud")
processos = datajud.listar_processos(
    tribunal="TJSP",
    assuntos=[6064, 6233, 7775, 10064, 10356, 12484, 12487, 12489, 14760],
    ano_ajuizamento=2024,
    paginas=1,
)
```


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"


**Carregando os dados a partir do repositório da disciplina:**


In [ ]:
saude = pd.read_csv(f"{URL}/tjsp_datajud_saude.csv")
saude.head()


In [ ]:
saude.shape


[Volta ao Índice](#indice)


___
<div id="tipos"></div>

# Tipos de variáveis

Antes de qualquer conta, é preciso **entender a natureza de cada variável**.
Essa classificação não está no arquivo: ela é uma decisão sua, e é ela que
determina que conta faz sentido.

🔹 **IDENTIFICADOR** <br>
> Serve para localizar o caso, não para medir nada. Nunca entra em conta.<br>
> *Exemplos:* `numero_processo`, `municipio_ibge`.

🔹 **CATEGÓRICA NOMINAL** <br>
> Rótulos sem nenhuma ordem natural entre si.<br>
> *Exemplos:* `sistema` (SAJ, Projudi), `classe` (Procedimento Comum, Execução).

🔹 **CATEGÓRICA ORDINAL** <br>
> Rótulos com ordem natural, mas sem distância mensurável entre eles.<br>
> *Exemplo:* `grau` (G1 vem antes de G2).

🔹 **CATEGÓRICA BINÁRIA** <br>
> Só dois valores possíveis. É um caso particular da nominal, e ganha nome
> próprio porque permite contas que as outras não permitem.<br>
> *Exemplo:* `formato` (eletrônico, físico).

🔹 **NUMÉRICA DISCRETA** <br>
> Resultado de **contar**. Assume valores inteiros e isolados.<br>
> *Exemplo:* `n_assuntos`.

🔹 **NUMÉRICA CONTÍNUA** <br>
> Resultado de **medir** numa escala. Entre dois valores sempre cabe outro.<br>
> *Exemplo:* nenhuma ainda nesta base, vamos criar uma.

🔹 **DATA** <br>
> Não é bem um tipo da lista: é matéria-prima. O que entra na análise é o que
> se calcula a partir dela.<br>
> *Exemplos:* `data_ajuizamento`, `data_ultima_atualizacao`.


<div id="ex1"></div>

### EXERCÍCIO 1

Classifique cada variável da base preenchendo o dicionário abaixo. As três
primeiras estão preenchidas como modelo.


In [ ]:
tipos = {
    # os três primeiros são o modelo
    "numero_processo": "identificador",
    "sistema": "categorica_nominal",
    "n_assuntos": "numerica_discreta",
    # complete daqui para baixo
    "tribunal": "________",
    "grau": "________",
    "classe": "________",
    "classe_codigo": "________",
    "assunto": "________",
    "orgao_julgador": "________",
    "municipio_ibge": "________",
    "formato": "________",
    "nivel_sigilo": "________",
    "data_ajuizamento": "data",
    "data_ultima_atualizacao": "data",
}

pd.Series(tipos).value_counts()


[Volta ao Índice](#indice)


___
<div id="primeiro-olhar"></div>

# O que o pandas achou de cada coluna

Ao ler o arquivo, o pandas atribui a cada coluna um **dtype**, que é o tipo de
armazenamento dele. Cuidado com a palavra "tipo": o dtype é um chute do pandas
a partir do formato do arquivo, e não a natureza da variável.


<div id="info"></div>

### 🔎 Olhando o dtype e a contagem de valores distintos


Mostra, para cada coluna, quantos valores não são nulos e qual o dtype.

✔️ **Uso do `.info()`**

```python
# Sintaxe geral:
DataFrame.info()
```

Documentação oficial: [.info()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html)


In [ ]:
saude.info()


Conta quantos valores distintos existem em cada coluna. É a segunda coisa a olhar, e já adianta muito sobre o tipo de cada variável.

✔️ **Uso do `.nunique()`**

```python
# Sintaxe geral:
DataFrame.nunique()
```

Documentação oficial: [.nunique()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nunique.html)


In [ ]:
saude.nunique()


<div id="dtype"></div>

### ⚠️ O dtype não é o tipo da variável

Estas quatro colunas vieram todas como número:


In [ ]:
saude[["classe_codigo", "municipio_ibge", "nivel_sigilo", "n_assuntos"]].dtypes


E o pandas deixa você fazer esta conta sem reclamar:


In [ ]:
saude["municipio_ibge"].mean()


Saiu um número, e ele não significa nada: `municipio_ibge` é um **código**, um
rótulo que por acaso é escrito com dígitos. A média de um rótulo é um número sem
referente no mundo. O mesmo vale para `classe_codigo`.

Já `n_assuntos` é uma contagem de verdade, e a média dela responde a alguma
coisa: quantos assuntos, em média, um processo tem. Mesma aparência no arquivo,
natureza diferente.

> 🤔 **Regra prática:** se somar dois valores da variável não produz nada com
> sentido, ela não é numérica, por mais que seja escrita com dígitos.


<div id="constante"></div>

### 🚩 Variável que não varia

Variável com um valor só não explica nada: ela é constante no recorte.


In [ ]:
saude[["tribunal", "nivel_sigilo", "formato"]].nunique()


In [ ]:
saude["formato"].value_counts()


`formato` é binária no papel, mas tem 3998 eletrônicos e 2 físicos.
Tecnicamente varia; na prática, não dá para comparar nada com dois casos de um
lado. Isso é decisão de pesquisa, não de programação.


[Volta ao Índice](#indice)


___
<div id="converter"></div>

# Convertendo o tipo de uma coluna

O pandas leu o arquivo do jeito que conseguiu. Agora cabe a você ajustar cada
coluna para que ela represente a variável que você identificou no Exercício 1.


<div id="astype"></div>

### 🔤 De número para texto


Converte a coluna para outro tipo. O método devolve uma **cópia**, então é preciso reatribuir o resultado à própria coluna. Fazemos isso com códigos, para que ninguém calcule média deles por acidente.

✔️ **Uso do `.astype()`**

```python
# Sintaxe geral:
DataFrame["coluna"] = DataFrame["coluna"].astype("string")
```

Documentação oficial: [.astype()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html)


In [ ]:
saude["classe_codigo"] = saude["classe_codigo"].astype("string")

saude["classe_codigo"].dtype


Com `municipio_ibge` tem um passo a mais. Ela veio como `float64` (repare no
`.0` no fim de cada número), porque o pandas usa `float` para poder representar
o valor faltante. Converter direto para texto guardaria o `.0` junto:


In [ ]:
saude["municipio_ibge"].head(3)


In [ ]:
saude["municipio_ibge"].astype("string").head(3)


A solução é passar antes por `"Int64"`, com I maiúsculo, que é o inteiro do
pandas que aceita valor faltante. Depois, sim, vira texto:


In [ ]:
saude["municipio_ibge"] = saude["municipio_ibge"].astype("Int64").astype("string")

saude["municipio_ibge"].head(3)


<div id="datas"></div>

### 📅 De texto para data

No arquivo, data é texto. Enquanto for texto, `2024-03-15` é só uma sequência de
caracteres: não dá para subtrair, nem ordenar direito, nem pedir o ano.


✔️ **Uso do `pd.to_datetime()`**

```python
# Sintaxe geral:
DataFrame["coluna"] = pd.to_datetime(DataFrame["coluna"])
```

Documentação oficial: [pd.to_datetime()](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)


In [ ]:
saude["data_ajuizamento"].head(3)


In [ ]:
saude["data_ajuizamento"] = pd.to_datetime(saude["data_ajuizamento"])

saude["data_ajuizamento"].head(3)


Repare no que mudou: o dtype passou de `object` para `datetime64[ns]`.


**✍️ Agora você.** Converta `data_ultima_atualizacao` do mesmo jeito.


In [ ]:
saude["data_ultima_atualizacao"] = pd.________(saude["data_ultima_atualizacao"])

saude[["data_ajuizamento", "data_ultima_atualizacao"]].dtypes


Data é matéria-prima: o que entra na análise é o que se calcula a partir dela. O acessador `.dt` dá acesso aos pedaços da data, como `.dt.year` (ano), `.dt.month` (mês) e `.dt.day` (dia).

✔️ **Uso do `.dt`**

```python
# Sintaxe geral:
DataFrame["coluna"].dt.year
```

Documentação oficial: [.dt](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.year.html)


In [ ]:
saude["ano_ajuizamento"] = saude["data_ajuizamento"].dt.year

saude[["data_ajuizamento", "ano_ajuizamento"]].head(3)


**✍️ Agora você.** Crie `mes_ajuizamento` com `.dt.month`.


In [ ]:
saude["mes_ajuizamento"] = saude["data_ajuizamento"].dt.________

saude[["data_ajuizamento", "ano_ajuizamento", "mes_ajuizamento"]].head(3)


`ano_ajuizamento` sai como número inteiro, e é assim que ele fica. O que muda de
uma análise para outra não é o tipo da coluna, e sim o uso: às vezes ele entra
como número (diferença entre dois anos), às vezes como agrupador (comparar 2023,
2024 e 2025 entre si). Vale escrever qual dos dois usos você está fazendo.


<div id="subtracao"></div>

### ➖ Subtraindo duas datas

Esta base não traz nenhuma variável de tempo pronta. A duração que a pergunta
pede precisa ser construída, e ela sai da subtração de duas datas.


Subtrair duas datas devolve um `Timedelta`, que é uma duração. Para virar número é preciso escolher a unidade, e `.dt.days` devolve a duração em dias.

✔️ **Uso do `.dt.days`**

```python
# Sintaxe geral:
(DataFrame["data_fim"] - DataFrame["data_inicio"]).dt.days
```

Documentação oficial: [.dt.days](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.days.html)


In [ ]:
(saude["data_ultima_atualizacao"] - saude["data_ajuizamento"]).head(3)


In [ ]:
saude["dias_ate_atualizacao"] = (
    saude["data_ultima_atualizacao"] - saude["data_ajuizamento"]
).dt.days

saude[["data_ajuizamento", "data_ultima_atualizacao", "dias_ate_atualizacao"]].head(3)


Pronto: a **primeira variável numérica contínua** da base, e ela não existia no
arquivo. Foi você que a criou.


Dá um resumo rápido de uma coluna numérica: contagem, média, desvio padrão, mínimo, quartis e máximo.

✔️ **Uso do `.describe()`**

```python
# Sintaxe geral:
DataFrame["coluna"].describe()
```

Documentação oficial: [.describe()](https://pandas.pydata.org/docs/reference/api/pandas.Series.describe.html)


In [ ]:
saude["dias_ate_atualizacao"].describe()


<div id="ex2"></div>

### EXERCÍCIO 2

Olhando o resultado de `.describe()` acima, responda em uma ou duas frases: o
tempo até a última movimentação parece bem distribuído, ou há sinal de casos
muito fora do padrão? Em que número você se baseou?


In [ ]:
# ESCREVA SUA RESPOSTA AQUI (em comentário ou em célula de texto)


[Volta ao Índice](#indice)


___
<div id="categoricas"></div>

# Variáveis categóricas

Categórica é o tipo mais comum no Direito e o mais chato de representar. Por
padrão o pandas guarda texto (`object`), o que funciona, mas perde duas coisas:
**quais são as categorias possíveis** e **se existe ordem entre elas**.


<div id="categorical"></div>

### 🏷️ Declarando a variável como categórica


Declara a coluna como categórica e guarda, junto dela, a lista de categorias possíveis.

✔️ **Uso do `pd.Categorical()`**

```python
# Sintaxe geral:
DataFrame["coluna"] = pd.Categorical(DataFrame["coluna"])
```

Documentação oficial: [pd.Categorical()](https://pandas.pydata.org/docs/reference/api/pandas.Categorical.html)


In [ ]:
saude["assunto"] = pd.Categorical(saude["assunto"])

saude["assunto"].dtype


Mostra os rótulos que a variável categórica conhece.

✔️ **Uso do `.cat.categories`**

```python
# Sintaxe geral:
DataFrame["coluna"].cat.categories
```

Documentação oficial: [.cat.categories](https://pandas.pydata.org/docs/reference/api/pandas.Series.cat.categories.html)


In [ ]:
saude["assunto"].cat.categories


**✍️ Agora você.** Faça o mesmo com a coluna `classe` e veja quantas categorias ela tem.


In [ ]:
saude["classe"] = pd.________(saude["classe"])

saude["classe"].cat.________


<div id="ordinal"></div>

### 🔢 Declarando a ordem das categorias

Quando existe ordem natural, ela precisa ser declarada: o pandas não adivinha.


`categories=` recebe os rótulos **na ordem certa** e `ordered=True` diz que essa ordem vale. `grau` tem G1 (primeiro grau) e G2 (segundo grau).

✔️ **Uso do `pd.Categorical(categories=, ordered=)`**

```python
# Sintaxe geral:
DataFrame["coluna"] = pd.Categorical(
    DataFrame["coluna"],
    categories=["menor", "meio", "maior"],
    ordered=True,
)
```

Documentação oficial: [pd.Categorical(categories=, ordered=)](https://pandas.pydata.org/docs/reference/api/pandas.Categorical.html)


In [ ]:
saude["grau_ordenado"] = pd.Categorical(
    saude["grau"], categories=["G1", "G2"], ordered=True
)

saude["grau_ordenado"].dtype


Com `ordered=True`, comparação e ordenação passam a funcionar: dá para comparar
a coluna com o nome de uma categoria usando `>`, `>=`, `<` e `<=`.


In [ ]:
(saude["grau_ordenado"] >= "G2").sum()


Agora repare no efeito colateral. A base tem um terceiro valor, `JE` (juizado
especial), que ficou de fora da lista de categorias:


In [ ]:
saude["grau"].unique()


In [ ]:
saude["grau_ordenado"].isna().sum()


> ⚠️ **Armadilha 1.** Todo valor fora da lista de categorias vira valor faltante,
> **em silêncio**. Isso é útil (declara o universo esperado) e perigoso (perde
> dado sem avisar). Confira sempre quantos viraram faltante depois de criar a
> categórica.

Aqui a perda é intencional: `JE` não fica nem antes nem depois de G1 e G2, é
outra coisa.


<div id="outros"></div>

### 🕳️ Valor faltante e a categoria Outros

A coluna `assunto` tem processos sem assunto informado. Juntar esses casos numa
categoria explícita costuma ser melhor do que deixá-los faltantes: a categoria
aparece nas contagens e ninguém esquece que ela existe.

A tentação é chamar `.fillna("Outros")` direto, mas isso levanta erro: `Outros`
não está na lista de categorias declaradas. É preciso abrir espaço para a
categoria antes.


Acrescenta rótulos à lista de categorias. Encadeando com `.fillna()`, os faltantes passam a cair na categoria nova.

✔️ **Uso do `.cat.add_categories()`**

```python
# Sintaxe geral:
DataFrame["coluna"] = (
    DataFrame["coluna"].cat.add_categories(["Outros"]).fillna("Outros")
)
```

Documentação oficial: [.cat.add_categories()](https://pandas.pydata.org/docs/reference/api/pandas.Series.cat.add_categories.html)


In [ ]:
saude["assunto"].isna().sum()


In [ ]:
saude["assunto"] = saude["assunto"].cat.add_categories(["Outros"]).fillna("Outros")

saude["assunto"].isna().sum()


In [ ]:
saude["assunto"].value_counts()


> ⚠️ **Armadilha 2.** `value_counts()` em categórica mostra **todas** as
> categorias declaradas, inclusive as com zero ocorrências. Ótimo para tabela (a
> linha existe mesmo com zero) e péssimo se você não esperava.


[Volta ao Índice](#indice)


___
<div id="cut"></div>

# ✂️ De numérica para categórica com pd.cut

Até aqui convertemos texto em data e número em texto. Falta o caminho de volta:
transformar uma variável **numérica** em **categórica ordinal**, agrupando os
valores em faixas.


`bins` são os pontos de corte e `labels` são os nomes das faixas. O resultado já sai como categórica **ordenada**.

✔️ **Uso do `pd.cut()`**

```python
# Sintaxe geral:
DataFrame["faixa"] = pd.cut(
    DataFrame["coluna_numerica"],
    bins=[limite1, limite2, limite3],
    labels=["faixa A", "faixa B"],
)
```

Documentação oficial: [pd.cut()](https://pandas.pydata.org/docs/reference/api/pandas.cut.html)


In [ ]:
saude["faixa_dias"] = pd.cut(
    saude["dias_ate_atualizacao"],
    bins=[-float("inf"), 30, 180, 365, float("inf")],
    labels=["até 1 mês", "1 a 6 meses", "6 a 12 meses", "mais de 1 ano"],
)

saude["faixa_dias"].dtype


In [ ]:
saude["faixa_dias"].value_counts(sort=False)


`sort=False` mantém a ordem das faixas. Sem ele, o pandas ordenaria pela
contagem, e a tabela perderia a sequência que interessa.

> 🤔 Cortar em faixas **perde informação**: 31 dias e 179 dias viram a mesma
> coisa. Faça isso quando a faixa for o que interessa para a pergunta, não por
> hábito.


<div id="ex3"></div>

### EXERCÍCIO 3

Como `faixa_dias` é ordenada, dá para comparar com o nome de uma faixa, do mesmo
jeito que fizemos com `grau_ordenado >= "G2"`. Conte quantos processos
demoraram mais de seis meses.


In [ ]:
(saude["faixa_dias"] ________ "1 a 6 meses").sum()


[Volta ao Índice](#indice)


___
<div id="resposta"></div>

# Respondendo à pergunta

Com os tipos arrumados, as contas ficam curtas. Quantos processos por assunto:


In [ ]:
saude["assunto"].value_counts()


E o tempo até a última movimentação, comparando primeiro e segundo grau. Por
enquanto fazemos isso separando a base em dois pedaços; na aula 4 você vai ver o
`groupby`, que faz esse tipo de comparação em uma linha só.


In [ ]:
primeiro_grau = saude[saude["grau"] == "G1"]

primeiro_grau["dias_ate_atualizacao"].median()


**✍️ Agora você.** Faça o mesmo para o segundo grau, criando `segundo_grau` do mesmo jeito.


In [ ]:
segundo_grau = saude[saude["grau"] == "________"]

segundo_grau["dias_ate_atualizacao"].________()


<div id="ex4"></div>

### EXERCÍCIO 4

A coluna `orgao_julgador` tem mais de mil valores distintos. Ela é categórica
nominal, mas com tantas categorias não serve para comparar grupos. Escreva, em
duas ou três linhas, que variável derivada dela você criaria para que ela
virasse útil, e por quê. Não precisa programar.


In [ ]:
# ESCREVA SUA RESPOSTA AQUI (em comentário ou em célula de texto)


[Volta ao Índice](#indice)


___
<div id="resumo"></div>

# RESUMO

O **dtype** é o chute do pandas a partir do formato do arquivo. O **tipo da
variável** é decisão sua, e é ele que determina que conta faz sentido. Ajustar
um ao outro é o trabalho desta aula.

Abaixo, todas as operações da aula em sequência, para consulta rápida.


In [ ]:
#=> LER A BASE
saude = pd.read_csv(f"{URL}/tjsp_datajud_saude.csv")

#=> OLHAR: dtype de cada coluna e quantos valores distintos
saude.info()
saude.nunique()

#=> NÚMERO -> TEXTO: para códigos, que não devem entrar em conta
saude["classe_codigo"] = saude["classe_codigo"].astype("string")

#=> NÚMERO COM FALTANTE -> TEXTO: passa por "Int64" para não carregar o ".0"
saude["municipio_ibge"] = saude["municipio_ibge"].astype("Int64").astype("string")

#=> TEXTO -> DATA
saude["data_ajuizamento"] = pd.to_datetime(saude["data_ajuizamento"])
saude["data_ultima_atualizacao"] = pd.to_datetime(saude["data_ultima_atualizacao"])

#=> DATA -> NÚMERO: pedaços da data com .dt
saude["ano_ajuizamento"] = saude["data_ajuizamento"].dt.year

#=> DUAS DATAS -> DURAÇÃO EM DIAS
saude["dias_ate_atualizacao"] = (
    saude["data_ultima_atualizacao"] - saude["data_ajuizamento"]
).dt.days

#=> TEXTO -> CATEGÓRICA NOMINAL
saude["assunto"] = pd.Categorical(saude["assunto"])

#=> TEXTO -> CATEGÓRICA ORDINAL: categories na ordem certa e ordered=True
saude["grau_ordenado"] = pd.Categorical(
    saude["grau"], categories=["G1", "G2"], ordered=True
)

#=> FALTANTE -> CATEGORIA "Outros": abre espaço antes de preencher
saude["assunto"] = saude["assunto"].cat.add_categories(["Outros"]).fillna("Outros")

#=> NUMÉRICA -> CATEGÓRICA ORDINAL: faixas com pd.cut
saude["faixa_dias"] = pd.cut(
    saude["dias_ate_atualizacao"],
    bins=[-float("inf"), 30, 180, 365, float("inf")],
    labels=["até 1 mês", "1 a 6 meses", "6 a 12 meses", "mais de 1 ano"],
)


**Duas armadilhas para levar daqui:**

1. Valor fora da lista de `categories` vira faltante **em silêncio**. Confira com
   `.isna().sum()` depois de criar uma categórica.
2. `value_counts()` em categórica mostra também as categorias com zero casos.

Na aula 3 vamos usar esses tipos para escolher a estatística certa.


[Volta ao Índice](#indice)
